In [3]:
import pandas as pd

df = pd.read_csv("Nat_gas.csv")
df["Dates"] = pd.to_datetime(df["Dates"], format="%m/%d/%y")
df = df.set_index("Dates")
df["Prices"] = df["Prices"].astype(float)

In [5]:
def price_storage_contract(
    price_series,
    injection_dates,
    withdrawal_dates,
    injection_rate,
    withdrawal_rate,
    max_storage,
    storage_cost_per_period,
    injection_cost_per_unit=0.0,
    withdrawal_cost_per_unit=0.0
):
    inventory = 0.0
    contract_value = 0.0

    # Ensure dates are sorted
    all_dates = sorted(price_series.index)

    for date in all_dates:
        price = price_series.loc[date]

        # Injection
        if date in injection_dates:
            inject_volume = min(injection_rate, max_storage - inventory)
            inventory += inject_volume

            contract_value -= inject_volume * price
            contract_value -= inject_volume * injection_cost_per_unit

        # Withdrawal
        if date in withdrawal_dates:
            withdraw_volume = min(withdrawal_rate, inventory)
            inventory -= withdraw_volume

            contract_value += withdraw_volume * price
            contract_value -= withdraw_volume * withdrawal_cost_per_unit

        # Storage cost (applied every period)
        contract_value -= inventory * storage_cost_per_period

    return contract_value
